# APIM 경유 Azure OpenAI 호출 테스트

이 노트북은 **APIM Subscription Key** 방식으로 Azure OpenAI를 호출하는 실습/검증용입니다.

- 학생 계정(Entra ID) 불필요
- 필요한 것: APIM Endpoint + Subscription Key
- 관련 문서: [apim-architecture-guide.md](./apim-architecture-guide.md), [apim-setup-guide.md](./apim-setup-guide.md)

## 체크리스트
1. `.env` 파일에 아래 4개 환경변수가 설정되어 있는지 확인
   - `APIM_BASE_URL`
   - `CHAT_MODEL`
   - `APIM_KEY`
   - `EMBEDDING_MODEL`
2. 네트워크에서 APIM 도메인(`*.azure-api.net`) 접근 가능한지 확인


## 0. 환경 준비

In [ ]:
# 필요한 패키지는 pyproject.toml에 이미 포함되어 있습니다.
# !pip install -q openai python-dotenv requests

import os
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True), override=True)

APIM_BASE_URL   = os.getenv("APIM_BASE_URL", "https://apim-foundryproxy-dev.azure-api.net/foundry").rstrip("/")
APIM_KEY        = os.getenv("APIM_KEY", "")
CHAT_MODEL      = os.getenv("CHAT_MODEL", "gpt-5.4")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")

assert APIM_KEY, "APIM_KEY 환경변수가 비어 있습니다. .env 파일을 확인하세요."

print("Base URL    :", APIM_BASE_URL)
print("Chat model  :", CHAT_MODEL)
print("Embed model :", EMBEDDING_MODEL)
print("API Key     :", APIM_KEY[:6] + "..." + APIM_KEY[-4:])


## 1. 연결 헬스체크 (raw HTTP)

SDK 없이 `requests`로 APIM 게이트웨이에 직접 호출해 네트워크·키·배포명을 한 번에 확인합니다.


In [ ]:
import requests, json

url = f"{APIM_BASE_URL}/{CHAT_MODEL}/chat/completions"
headers = {
    "Content-Type": "application/json",
    "api-key": APIM_KEY,
}
payload = {
    "messages": [{"role": "user", "content": "ping"}],
    "max_completion_tokens": 5,
}

r = requests.post(url, headers=headers, json=payload, timeout=30)
print("HTTP", r.status_code)
try:
    print(json.dumps(r.json(), ensure_ascii=False, indent=2)[:800])
except Exception:
    print(r.text[:800])

assert r.status_code == 200, "헬스체크 실패: APIM_BASE_URL / APIM_KEY / CHAT_MODEL을 확인하세요."


## 2. OpenAI SDK로 Chat Completions

Foundry Proxy APIM은 `api-key` 헤더로 인증합니다.
`default_headers`로 주입하고 `base_url`은 `{APIM_BASE_URL}/{model}/` 형태로 모델별로 분리합니다.


In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="placeholder",                       # APIM은 헤더로 인증
    base_url=f"{APIM_BASE_URL}/{CHAT_MODEL}/",
    default_headers={"api-key": APIM_KEY},
)

resp = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "너는 친절한 한국어 AI 튜터야."},
        {"role": "user",   "content": "대한민국의 수도를 한 문장으로 알려줘."},
    ],
    max_completion_tokens=80,
    temperature=0.3,
)

print(resp.choices[0].message.content)
print("---")
print("usage:", resp.usage)


## 3. 스트리밍 응답

In [ ]:
stream = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": "파이썬으로 피보나치 수열을 1줄로 표현해줘."}],
    max_completion_tokens=120,
    stream=True,
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()


## 4. 임베딩(Embeddings) 테스트

In [ ]:
embedding_client = OpenAI(
    api_key="placeholder",
    base_url=f"{APIM_BASE_URL}/{EMBEDDING_MODEL}/",
    default_headers={"api-key": APIM_KEY},
)

try:
    emb = embedding_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=["APIM은 게이트웨이입니다.", "Foundry Proxy로 인증을 대행합니다."],
    )
    print("벡터 개수:", len(emb.data))
    print("벡터 차원:", len(emb.data[0].embedding))
    print("usage   :", emb.usage)
except Exception as e:
    print("임베딩 호출 실패:", e)


## 5. 보안/정책 검증

APIM 정책이 기대대로 동작하는지 확인합니다.


### 5-1. 잘못된 Subscription Key → 401

In [ ]:
r = requests.post(
    url,
    headers={"Content-Type": "application/json", "api-key": "invalid-key-xxx"},
    json=payload,
    timeout=30,
)
print("HTTP", r.status_code)
print(r.text[:300])
assert r.status_code in (401, 403), "APIM이 잘못된 키를 거부하지 못했습니다."
print("✅ 잘못된 키 거부 확인")


### 5-2. Subscription Key 누락 → 401

In [ ]:
r = requests.post(
    url,
    headers={"Content-Type": "application/json"},
    json=payload,
    timeout=30,
)
print("HTTP", r.status_code)
print(r.text[:300])
assert r.status_code in (401, 403), "APIM이 키 없는 요청을 거부하지 못했습니다."
print("✅ 키 누락 거부 확인")


### 5-3. `api-key` 헤더만 보내도 무시되는지

학생이 실수로/의도적으로 원본 `api-key`를 넣어도 APIM 정책이 제거합니다.
Subscription Key 없이는 통과하면 안 됩니다.


In [ ]:
r = requests.post(
    url,
    headers={"Content-Type": "application/json", "Ocp-Apim-Subscription-Key": "whatever"},
    json=payload,
    timeout=30,
)
print("HTTP", r.status_code)
assert r.status_code in (401, 403), "Ocp-Apim-Subscription-Key 단독으로 통과됨 → 정책 점검 필요"
print("✅ 옛 헤더 단독 호출 거부 확인")


## 6. 간단 벤치마크 (선택)

평균 지연시간과 실패율을 빠르게 확인합니다. 남용 방지를 위해 10회로 제한합니다.


In [ ]:
import time, statistics

N = 10
latencies = []
errors = 0

for i in range(N):
    t0 = time.perf_counter()
    try:
        client.chat.completions.create(
            model=CHAT_MODEL,
            messages=[{"role": "user", "content": f"숫자 {i}를 한국어로 써줘."}],
            max_completion_tokens=10,
        )
        latencies.append(time.perf_counter() - t0)
    except Exception as e:
        errors += 1
        print(f"[{i}] error:", e)

if latencies:
    print(f"성공 {len(latencies)}/{N}")
    print(f"평균 지연: {statistics.mean(latencies):.2f}s")
    print(f"최소/최대: {min(latencies):.2f}s / {max(latencies):.2f}s")
print(f"실패: {errors}")


## 7. 트러블슈팅

| 증상 | 원인 | 해결 |
|------|------|------|
| 401 Access denied due to invalid subscription key | Subscription Key 오타/비활성 | 운영자에게 키 재발급 요청 |
| 404 Resource not found | `deployment-id` 오타 또는 허용 목록 외 모델 | `EMBEDDING_MODEL` 확인 |
| 403 Forbidden (IP) | 교육장 외부에서 호출 | IP allowlist 정책 확인 |
| 429 Too Many Requests | 분당/일간 쿼터 초과 | 잠시 후 재시도 또는 운영자 문의 |
| 500/502 | 백엔드/MI 토큰 문제 | `apim-setup-guide.md` Step 3, 6 재점검 |
| SDK: api_key required | SDK가 빈 키를 거부 | `api_key="unused"` 같은 더미 문자열 지정 |
